## **EDA & Pre-modeling diagnostics**
- **Statistical summary** of daily log-returns for 5 currency pairs: USD/PHP, CNY/PHP, JPY/PHP, HKD/PHP, and SGD/PHP
- **Objectives:** Verify distribution (normality), serial correlation (linear), and ARCH effects (non-linear)
- **Context:** Supplements ADF/KPSS/Bai-Perron tests for high-fidelity model justification


In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.api as sm
import yaml
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

warnings.filterwarnings("ignore")

# 1. Coordinate Paths: Move to project root if running from notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")


In [2]:
# 1. Load context from central config
with open("configs/pipeline_config.yaml", "r") as f:
    config = yaml.safe_load(f)
active_target = config["active_target"]
processed_path = os.path.join(
    # "..",
    config["paths"]["processed"],
    active_target,
    "fx_aligned.csv",
)
# 2. Load dynamic target data
df_train = pd.read_csv(processed_path, parse_dates=["Date"])
df_train.set_index("Date", inplace=True)
# 3. Auto-detect return columns based on the current target basket
ret_cols = [c for c in df_train.columns if c.endswith("_RET")]
# Construct readable labels (e.g., 'USDSGD_RET' -> 'USD/SGD')
pairs_labels = [f"{c[:3]}/{c[3:6]}" for c in ret_cols]
results = {}
# 4. Modular Diagnostics Loop
for col, label in zip(ret_cols, pairs_labels):
    series = df_train[col].dropna()
    # Jarque-Bera (Normality)
    jb_stat, jb_p = stats.jarque_bera(series)
    # Ljung-Box (Autocorrelation at Lag 10)
    lb_p = acorr_ljungbox(series, lags=[10])["lb_pvalue"].iloc[0]
    # ARCH-LM (Volatility Clustering at Lag 10)
    # Note: Require at least 11 observations for nlags=10
    _, arch_p, _, _ = het_arch(series, nlags=10)
    results[label] = {
        "Count": series.count(),
        "Mean": series.mean(),
        "Std. Dev.": series.std(),
        "Min": series.min(),
        "Median": series.median(),
        "Max": series.max(),
        "IQR": series.quantile(0.75) - series.quantile(0.25),
        "Skewness": series.skew(),
        "Kurtosis (Excess)": series.kurtosis(),
        "Jarque-Bera (p-val)": jb_p,
        "Ljung-Box (p-val)": lb_p,
        "ARCH-LM (p-val)": arch_p,
    }


In [3]:
# 5. Output Table
pd.options.display.float_format = "{:.4f}".format
diag_table = pd.DataFrame(results)

print("Table 1: Descriptive Stats & Diagnostic Tests")
display(diag_table)


Table 1: Descriptive Stats & Diagnostic Tests


,USD/PHP,CNY/PHP,JPY/PHP,HKD/PHP,SGD/PHP
Count,4230.0000,4230.0000,4230.0000,4230.0000,4230.0000
Mean,0.0064,0.0062,-0.0065,0.0062,0.0084
Std. Dev.,0.4688,0.4984,0.7215,0.4657,0.4834
Min,-3.1305,-2.9488,-4.6903,-3.0924,-3.2763
Median,0.0000,0.0004,-0.0234,0.0028,0.0041
Max,4.1741,4.3383,3.7202,4.1708,3.9210
IQR,0.4639,0.4813,0.7855,0.4605,0.4984
Skewness,0.2324,0.2676,0.1170,0.2364,0.0864
Kurtosis (Excess),8.9053,7.8838,2.9896,9.0202,7.7612
Jarque-Bera (p-val),0.0000,0.0000,0.0000,0.0000,0.0000


- **Non-Normality (JB Test):** Null hypothesis rejected ($p < 0.01$). Excess Kurtosis ($\approx 3.0 - 9.1$) confirms a "heavy-tailed" leptokurtic distribution. This justifies non-parametric models (SVR/MLP) which absorb non-Gaussian error terms.
- **Serial Correlation (LB Test):** Significant linear dependency detected ($p < 0.01$). Log-returns show clear autocorrelation, validating the Autoregressive (ARIMA) and Vector (VAR) stage.
- **Volatility Clustering (ARCH-LM):** Strong non-linear dependency ($p < 0.01$). Predictable variance shifts (heteroscedasticity) justify the Hybrid model’s secondary stage.
- **p-value = 0.0000:** A numerical result of the high test statistics; the probability under a null hypothesis is beyond float precision ($< 10^{-16}$). It's not absolute zero but it's very small, 4 decimals isn't enough to show no-zero values.


### **I don't want to play with ts anymore**

In [4]:
df_plot = df_train.copy()
# 2. Extract pairs and return columns
price_cols = [c for c in df_plot.columns if not c.endswith("_RET")]
ret_cols = [c for c in df_plot.columns if c.endswith("_RET")]
labels = [f"{c[:3]}/{c[3:6]}" for c in ret_cols]
# 3. Setup Figure (2 rows, 1 column)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        f"<b>Raw Exchange Rates (Target: {active_target})</b>",
        "<b>Log-Returns with Thresholds</b>",
    ),
)
# Color palette for consistency across subplots
colors = ["#2c3e50", "#e74c3c", "#2980b9", "#27ae60", "#f39c12", "#8e44ad"]
# 4. Add Traces
for i, (p_col, r_col, label) in enumerate(zip(price_cols, ret_cols, labels)):
    color = colors[i % len(colors)]

    # Subplot 1: Raw Prices
    fig.add_trace(
        go.Scatter(
            x=df_plot.index,
            y=df_plot[p_col],
            name=label,
            line=dict(width=1.5, color=color),
            legendgroup=label,
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    # Subplot 2: Log Returns
    fig.add_trace(
        go.Scatter(
            x=df_plot.index,
            y=df_plot[r_col],
            name=label,
            line=dict(width=1, color=color),
            legendgroup=label,
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    # Optional: Add mean/quantile reference lines for the returns?
    # We will add them globally or per series if requested.
    # For now, let's calculate global 5/95% across the entire series to show tail risks
    q_low, q_high = df_plot[r_col].quantile([0.05, 0.95])

    # Add horizontal quantile markers (Subtle dashed lines)
    # We add these only to the second subplot
    fig.add_hline(
        y=q_low,
        line=dict(color=color, width=0.5, dash="dot"),
        opacity=0.3,
        row=2,
        col=1,
    )
    fig.add_hline(
        y=q_high,
        line=dict(color=color, width=0.5, dash="dot"),
        opacity=0.3,
        row=2,
        col=1,
    )
# 5. Add Subtle Background Shading for Splits
# Retrieve split dates from config if they exist
# For example: test_start = pd.to_datetime(config["splits"]["test_start"])
# Here we add a placeholder shading logic:
try:
    # 1. Retrieve Ratios from config['data']
    train_ratio = config["data"].get("train_ratio", 0.8)
    val_ratio = config["data"].get("val_ratio", 0.1)

    # 2. Calculate indices based on dataframe length
    N = len(df_plot)
    train_split_idx = int(N * train_ratio)
    test_split_idx = int(N * (train_ratio + val_ratio))

    # 3. Get exact timestamps from the index
    train_end = df_plot.index[train_split_idx]
    test_start = (
        df_plot.index[test_split_idx] if test_split_idx < N else df_plot.index[-1]
    )

    # --- Shading A: Training Zone ---
    fig.add_vrect(
        x0=df_plot.index[0],
        x1=train_end,
        fillcolor="#95a5a6",
        opacity=0.12,
        layer="below",
        line_width=0,
        annotation_text="<b>TRAIN</b>",
        annotation_position="top left",
        annotation_font=dict(size=10, color="#7f8c8d"),
        row="all",
        col=1,
    )

    # --- Shading B: Validation Zone (Slightly different tint) ---
    fig.add_vrect(
        x0=train_end,
        x1=test_start,
        fillcolor="#bdc3c7",
        opacity=0.06,
        layer="below",
        line_width=0,
        annotation_text="<b>VAL</b>",
        annotation_position="top left",
        annotation_font=dict(size=9, color="#95a5a6"),
        row="all",
        col=1,
    )
    # --- Shading C: Test Zone ---
    fig.add_vrect(
        x0=test_start,
        x1=df_plot.index[-1],
        fillcolor="#3498db",
        opacity=0.08,
        layer="below",
        line_width=0,
        annotation_text="<b>TEST</b>",
        annotation_position="top left",
        annotation_font=dict(size=10, color="#2980b9"),
        row="all",
        col=1,
    )

    # Vertical boundary markers
    fig.add_vline(
        x=train_end, line_width=1, line_dash="dot", line_color="#7f8c8d", opacity=0.8
    )
    fig.add_vline(
        x=test_start, line_width=1, line_dash="dot", line_color="#2980b9", opacity=0.8
    )
except Exception as e:
    print(
        f"Shading failed: Ensure df_plot index is datetime and config['data'] ratios exist. Error: {e}"
    )
# 6. Final Polish & Aesthetics
fig.update_layout(
    template="plotly_white",
    height=800,
    width=1200,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(l=60, r=60, t=120, b=60),
)
fig.update_xaxes(showgrid=True, gridcolor="#f0f0f0")
fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0", zerolinecolor="#bdc3c7")
fig.show()


### **Distribution Analysis (USD/PHP)**
- Visually inspect **fat tails** and **non-normality**
- **Histogram**: Frequency distribution with KDE overlay
- **Q-Q Plot**: Deviations from theoretical Normal distribution


In [5]:
# 1. Data Preparation
val = df_train[f"USD{active_target}_RET"].dropna()
# Calculate Q-Q points
(osm, osr), (slope, intercept, r) = stats.probplot(val, dist="norm")
line_x = np.array([osm.min(), osm.max()])
line_y = slope * line_x + intercept


In [6]:
# 2. Dynamic Range Control: 
if active_target == "PHP":
    h_range = [-4.3, 4.3]        # Returns % (X-axis Dist, Y-axis QQ)
    v_range_dist = [0, 2]  # Peak Density (Y-axis Dist)
    q_range = [-5, 5]        # Theoretical Quantiles (X-axis QQ)
else:
    h_range = [-9, 9]
    v_range_dist = [0, 12]
    q_range = [-6, 6]

# 3. Setup Figure
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("<b>Return Distribution</b>", "<b>Quantile-Quantile Plot</b>"),
    horizontal_spacing=0.12,
)
# --- Subplot 1: Distribution ---
fig.add_trace(
    go.Histogram(
        x=val,
        nbinsx=500,  # High granularity for the peak
        histnorm="probability density",
        name="Empirical",
        marker_color="#34495e",  # Slightly lighter slate
        opacity=0.85,
    ),
    row=1,
    col=1,
)
# Normal Curve (Reference)
x_range = np.linspace(h_range[0], h_range[1], 1000)
y_norm = stats.norm.pdf(x_range, val.mean(), val.std())
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=y_norm,
        mode="lines",
        name="Normal Approx",
        line=dict(color="#e74c3c", width=2, dash="dot"),
    ),
    row=1,
    col=1,
)
# --- Subplot 2: Q-Q Plot ---
fig.add_trace(
    go.Scatter(
        x=osm,
        y=osr,
        mode="markers",
        name="Quantiles",
        marker=dict(color="#d35400", size=3.5, opacity=0.4),
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        name="Ideal Normal",
        line=dict(color="#2c3e50", width=1.5),
    ),
    row=1,
    col=2,
)
# 4. Global Refinement
fig.update_layout(
    template="plotly_white",
    height=550,
    width=1200,  # Slightly wider
    showlegend=False,
    bargap=0.05,  # Adds spacing between bins for "The Economist" look
    font=dict(family="Arial, sans-serif", size=13, color="#2c3e50"),
    title_text=f"<b>USD/{active_target} Log-Return Diagnostics</b>",
    title_font_size=20,
    title_x=0.05,
    margin=dict(t=100, b=80, l=80, r=80),
)
# Precision Axis Control
fig.update_xaxes(
    title_text="Daily Return (%)",
    range=h_range,
    row=1,
    col=1,
    showgrid=True,
    gridcolor="#f0f0f0",
    zerolinecolor="#bdc3c7",
)
fig.update_yaxes(title_text="Density", row=1, col=1, showgrid=True, gridcolor="#f0f0f0")
fig.update_xaxes(title_text="Theoretical Quantiles", row=1, col=2, gridcolor="#f0f0f0")
fig.update_yaxes(
    title_text="Ordered Return (%)",
    range=h_range,
    row=1,
    col=2,
    showgrid=True,
    gridcolor="#f0f0f0",
    zerolinecolor="#bdc3c7",
)
fig.show()


- **Leptokurtic Peaking:** The vertical spike at zero confirms a managed regime, making the distribution reach a peak density near **2.0** (significantly higher than the normal approximation of ~0.85).
- **Tail Divergence ($Q-Q$ Plot):** The strong "S-curve" reaching **$\pm 4\%$** indicates that extreme market jumps are significantly more frequent than predicted by a Normal distribution.
- **Model Justification:** The visual rejection of a Gaussian profile confirms that linear models will suffer from oversized residuals, requiring the help of non-linear Machine Learning.


### **Temporal Dependency Analysis**
- **ACF (Log-Returns):** Visual confirmation of linear autocorrelation (justifies ARIMA/VAR)
- **ACF (Squared Returns):** Visual proof of second-moment dependency (justifies Hybrid/ML logic)


In [7]:
# Calculate ACF values
series = df_train[f"USD{active_target}_RET"].dropna()
acf_vals = sm.tsa.acf(series, nlags=40)
acf_sq_vals = sm.tsa.acf(series**2, nlags=40)
lags = np.arange(len(acf_vals))


In [8]:
# Setup Subplots
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "<b>ACF: Log-Returns</b>",
        "<b>ACF: Squared Returns (Volatility)</b>",
    ),
    horizontal_spacing=0.12,
)

# Plot ACF
fig.add_trace(
    go.Bar(x=lags, y=acf_vals, marker_color="#2c3e50", name="ACF"), row=1, col=1
)
# Plot Squared ACF
fig.add_trace(
    go.Bar(x=lags, y=acf_sq_vals, marker_color="#d35400", name="Sq ACF"), row=1, col=2
)

# Significance Threshold (approx 95% CI)
ci = 1.96 / np.sqrt(len(series))
for col in [1, 2]:
    fig.add_hline(
        y=ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )
    fig.add_hline(
        y=-ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )

# Styling
fig.update_layout(
    template="plotly_white",
    height=450,
    width=1100,
    showlegend=False,
    title_text=f"<b>Auto-Correlation Structure (USD/{active_target})</b>",
    font=dict(family="Arial, sans-serif", size=13),
    margin=dict(t=80, b=60, l=60, r=60),
)

fig.update_xaxes(title_text="Lag", gridcolor="#f2f2f2")
fig.update_yaxes(title_text="Correlation", gridcolor="#f2f2f2")

fig.show()


### **Core Time-Series Dependencies (USD/VND)**
- **ACF:** Detects Moving Average (MA) components and general periodicity
- **PACF:** Isolates direct Autoregressive (AR) dependencies, stripping intermediate effects
- **Squared ACF:** Confirms second-moment dependency (volatility clustering) for Hybrid justification


In [9]:
from statsmodels.tsa.stattools import acf, pacf

# 1. Prepare 3-way diagnostics
series = df_train[f"USD{active_target}_RET"].dropna()
lags_to_show = 40

acf_vals = acf(series, nlags=lags_to_show)
pacf_vals = pacf(series, nlags=lags_to_show)
acf_sq_vals = acf(series**2, nlags=lags_to_show)
lags = np.arange(len(acf_vals))

# 2. Setup Subplots
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "<b>ACF (MA order)</b>",
        "<b>PACF (AR order)</b>",
        "<b>Sq-ACF (Volatility)</b>",
    ),
    horizontal_spacing=0.08,
)

# Plots
fig.add_trace(go.Bar(x=lags, y=acf_vals, marker_color="#2c3e50"), row=1, col=1)
fig.add_trace(go.Bar(x=lags, y=pacf_vals, marker_color="#34495e"), row=1, col=2)
fig.add_trace(go.Bar(x=lags, y=acf_sq_vals, marker_color="#d35400"), row=1, col=3)

# Significance Threshold (95% CI)
ci = 1.96 / np.sqrt(len(series))
for col in [1, 2, 3]:
    fig.add_hline(
        y=ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )
    fig.add_hline(
        y=-ci, line_dash="dash", line_color="#e74c3c", opacity=0.5, row=1, col=col
    )

# 3. Styling
fig.update_layout(
    template="plotly_white",
    height=400,
    width=1250,
    showlegend=False,
    title_text="<b>USD/VND Temporal Dependency Structure</b>",
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(t=80, b=50, l=50, r=50),
)
fig.update_xaxes(title_text="Lag")
fig.update_yaxes(
    range=[-1, 1], gridcolor="#f2f2f2"
)  # Standardize scale to see magnitude

fig.show()


- **Mean Reversion (ACF):** A massive negative spike at **Lag 1 (~ -0.25)** indicates an inverse 1-day memory. This suggests immediate price corrections typical of high-frequency mean-reversion.
- **Predictable PACF:** Significant negative spike at **Lag 1** suggests a direct AR(1) component, providing initial parameters for the ARIMA stage.
- **Volatility Continuity ($Sq-ACF$):** Positive, significant spikes (starting at **Lag 1**) prove that return magnitude (volatility) is persistent and highly predictable ($p < 0.01$).


**Cross-Comparison**
*   **Consistency:** The **Kurtosis (~9.0)** from the table is perfectly visualized by the distribution peak and the Q-Q tail divergence.
*   **Linearity vs. Volatility:** While the **Log-Returns (ACF)** show immediate mean-reversion at Lag 1, the **Squared Returns (Sq-ACF)** are persistent, proving that intensity is more predictable than direction.
*   **The Lag 1 Signature:** The dominance of **Lag 1** across ACF and Sq-ACF points to a structural immediate volatility response in the USD/PHP data, which the Hybrid model is specifically designed to exploit.
